In [1]:
import sys
print(sys.executable)

c:\Users\Pratham\OneDrive\Desktop\Portfolio\handwriting-ml\.venv\Scripts\python.exe


In [2]:
import random
import torch
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from torchvision import transforms
from PIL import Image
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv("writer_index.csv")

writer_map = {}

for writer_id, group in df.groupby("writer_id"):
    writer_map[writer_id] = group["filepath"].tolist()

In [ ]:
your_transforms = transforms.Compose([
    transforms.Resize((224, 224)),   # Resize all images
    transforms.ToTensor(),            # Convert PIL Image to Tensor
    transforms.Normalize(mean=[0.5], std=[0.5])  # Normalize grayscale image
])

In [ ]:
class SiameseDataset(Dataset):
    def __init__(self, writer_map, transform=None, num_samples=100000):
        """
        writer_map: dict -> {writer_id: [img_path1, img_path2, ...]}
        transform: torchvision transforms
        num_samples: virtual dataset size (since pairs are generated dynamically)
        """
        self.writer_map = writer_map
        self.transform = transform
        self.num_samples = num_samples

        self.writers = list(writer_map.keys())

        # optional sanity filter: remove writers with <2 images
        self.writers = [w for w in self.writers if len(writer_map[w]) >= 2]

    def __len__(self):
        return self.num_samples

    def _load_image(self, path):
        img = Image.open(path).convert("L")  # grayscale is typical for handwriting
        if self.transform:
            img = self.transform(img)
        return img

    def __getitem__(self, idx):
        # decide: positive or negative pair
        same = random.random() < 0.5

        if same:
            writer = random.choice(self.writers)
            img1, img2 = random.sample(self.writer_map[writer], 2)
            label = 1
        else:
            w1, w2 = random.sample(self.writers, 2)
            img1 = random.choice(self.writer_map[w1])
            img2 = random.choice(self.writer_map[w2])
            label = 0

        img1 = self._load_image(img1)
        img2 = self._load_image(img2)

        return img1, img2, torch.tensor(label, dtype=torch.float32)

In [ ]:
dataset = SiameseDataset(
    writer_map=writer_map,
    transform=your_transforms,
    num_samples=100000
)

In [ ]:
loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True
)

In [ ]:
img1, img2, label = dataset[0]

print(img1.shape)
print(img2.shape)
print(label)

In [ ]:
for img1, img2, labels in loader:
    print(img1.shape)
    print(img2.shape)
    print(labels.shape)
    break

In [ ]:
plt.subplot(1,2,1)
plt.imshow(img1.squeeze(), cmap="gray")
plt.title("Image 1")

plt.subplot(1,2,2)
plt.imshow(img2.squeeze(), cmap="gray")
plt.title("Image 2")

plt.suptitle(f"Label = {label.item()}")

plt.show()